# TAMER OCR v2.2 — Unified Execution Orchestrator

**Minimalist Runner Strategy:**
1. **Purge:** Zero-leftover environment cleanup.
2. **Auth:** HuggingFace and Kaggle credential setup.
3. **Clone:** Pull the latest production code from GitHub.
4. **Execute:** Trigger the strict pipeline (Preprocessing -> Sync -> Train).
5. **Artifacts:** Sync final best model to HuggingFace Hub.

### 1. Environment Purge & Directory Setup

In [ ]:
import os
import shutil
import gc

print("🧹 Cleaning workspace for a fresh start...")
directories = [
    'AI_PROJECT_TAMER_Complete', 
    'data', 
    'outputs', 
    'checkpoints', 
    'logs'
]

IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.chdir(WORK_DIR)

for d in directories:
    path = os.path.join(WORK_DIR, d)
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"  - Removed: {path}")

gc.collect()
print("✅ Workspace is clean.")

### 2. Authentication & Credentials

In [ ]:
import getpass

print("🔑 Configure credentials:")
hf_token = os.getenv('HF_TOKEN') or getpass.getpass('HuggingFace Token (Write Access): ')
os.environ['HF_TOKEN'] = hf_token

kaggle_user = 'merselfares'
kaggle_key = os.getenv('KAGGLE_KEY') or getpass.getpass('Kaggle API Key: ')
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = kaggle_user, kaggle_key

kaggle_conf = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_conf, exist_ok=True)
with open(os.path.join(kaggle_conf, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_user}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_conf, 'kaggle.json'), 0o600)

print("✅ Auth configured.")

### 3. Dependencies & Code Retrieval

In [ ]:
print("📦 Installing essential packages...")
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless

print("\n📥 Cloning production codebase...")
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
!git clone {REPO_URL}

print("✅ Codebase retrieved.")

### 4. Initialization & System Path

In [ ]:
import sys
from huggingface_hub import HfApi

REPO_PATH = os.path.join(WORK_DIR, 'AI_PROJECT_TAMER_Complete')
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)

from tamer_ocr.config import Config
config = Config()

# Map directories to environment-specific paths
config.data_dir = os.path.join(WORK_DIR, 'data')
config.output_dir = os.path.join(WORK_DIR, 'outputs')
config.checkpoint_dir = os.path.join(WORK_DIR, 'checkpoints')
config.log_dir = os.path.join(WORK_DIR, 'logs')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir, config.log_dir]: 
    os.makedirs(d, exist_ok=True)

# Fetch username for Hub logic
hf_api = HfApi(token=os.environ['HF_TOKEN'])
username = hf_api.whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{username}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{username}/tamer-preprocessed"

print(f"✅ Configured for Hub: {config.hf_repo_id}")

### 5. Launch Training Pipeline

In [ ]:
from tamer_ocr.core.trainer import Trainer

print("🚀 Launching TAMER Core Training...")
trainer = Trainer(config)
trainer.run() # Includes Preprocessing check, auto-resume, and training loop.

### 6. Final Failsafe Hub Sync

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

best_pt = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_pt):
    print("📤 Final push of best model to HuggingFace Hub...")
    push_checkpoint_to_hf(best_pt, config, epoch=0, is_best=True)
    print("✅ Sync complete.")
else:
    print("⚠️ Training concluded without producing best.pt artifact.")